In [1]:
# ===========================================================================
# ACOUSTIC FEATURE ANALYSIS - interactive driver

%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from src import config, eda
from src.console import print_header, print_kv, architecture_table
from src.preprocessing import (_load_resampled, mfcc_frame_count, build_mfcc_transform,
                               extract_mfcc_features_cached)
from src.praat import extract_suprasegmental_sequence, extract_segmental_extra_sequence
from src.style import apply_style, SEQUENTIAL_CMAP, FORMANT_COLORS, BRANCH_COLORS
from src.training.data import load_manifest
from src.training.models import SEVERITY_MODEL_NAME, build_model
from src.training.reporting import feature_audit
from src.vad import apply_vad

apply_style()
config.ensure_directories()

df_m6 = load_manifest()

print_header("Acoustic Feature Analysis")
print_kv("Manifest", config.MANIFEST_PATH)
print_kv("Utterances", len(df_m6))



══════════════════════════════════════════════════════════════════════════════
  ACOUSTIC FEATURE ANALYSIS
══════════════════════════════════════════════════════════════════════════════
  Manifest ................................ C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\m6_manifest.csv
  Utterances .............................. 21420


In [2]:
# STAGE 2 - VAD validation: raw waveform, detected speech region, the resulting VAD-processed waveform, and...
from src.visualization import plot_vad_validation_examples

vad_validation_path, example_stats = plot_vad_validation_examples(df_m6, seed=config.DEFAULT_SEED)
example_stats


c:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\src\visualization.py:167: UserWarning: Setting the 'color' property will override the edgecolor or facecolor properties.
  axes_row[3].axvspan(valid_frames - 0.5, total_frames - 0.5, color="0.15", alpha=0.55,



══════════════════════════════════════════════════════════════════════════════
  VAD VALIDATION
══════════════════════════════════════════════════════════════════════════════
  Examples (5 VAD-behaviour categories) ... 5
  Combined figure ......................... C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\figures\vad_validation_examples.png
  Per-example figures ..................... C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\figures/vad_validation_<category>_<speaker>_<file>.png
  [ ✗ ] 2919/21420 sampled utterances fell back to the original (untrimmed) waveform


,Category,Speaker_ID,Severity,original_duration_s,speech_duration_s,speech_ratio,num_segments,mfcc_valid_frames,mfcc_padded_frames,mfcc_padding_ratio,fallback_used
0,normal duration,M09,High,2.827750,0.828,0.292812,1,83,318,0.793017,False
1,long trailing silence,M01,Very Low,23.691688,0.668,0.028196,1,67,334,0.832918,False
2,long leading silence,M01,Very Low,33.614750,0.252,0.007497,1,26,375,0.935162,False
3,internal pauses,M10,High,53.797312,22.512,0.418460,20,401,0,0.000000,False
4,weak/low-energy dysarthric,M01,Very Low,33.614750,0.252,0.007497,1,26,375,0.935162,False


In [3]:
# STAGE 3 - Extract all FEATURE_COLUMNS Praat features for every utterance in the manifest: pitch        f0...
from src.praat import extract_praat_features_batch

praat_features = extract_praat_features_batch(df_m6, cache_path=config.PRAAT_FEATURES_PATH)
praat_features.head()


─── Praat acoustic analysis — 31 features x 21,420 utterances ────────────────
  Extracting Praat features        100%|███████████████████| 21420/21420 [1:54:46<00:00,  3.11utt/s]
  [ ✓ ] 21,420/21,420 utterances measured
  Praat features cached ................... C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\praat_features.csv


,Filename,Speaker_ID,Group,Severity,f0_mean,f0_max,f0_min,f0_std,f0_range,jitter_local,...,f2_std,f3_std,f2_f1_ratio,intensity_mean,intensity_max,intensity_min,intensity_std,speech_rate,pause_duration,voice_breaks
0,CF02_B1_C10_M6.wav,CF02,Healthy Control,N/A (Control),193.497376,240.973411,170.310333,20.434884,70.663079,0.012570,...,258.920020,473.127276,2.509759,43.242867,51.809923,25.451223,9.140189,57.978868,1.284713,0
1,CF02_B1_C11_M6.wav,CF02,Healthy Control,N/A (Control),192.606827,241.274043,172.630000,16.799769,68.644043,0.015317,...,162.486774,495.541012,2.600568,44.632957,52.393705,25.874688,9.620817,52.382685,1.503148,1
2,CF02_B1_C12_M6.wav,CF02,Healthy Control,N/A (Control),202.926716,437.321450,101.424637,83.372793,335.896813,0.031539,...,121.665290,240.332375,2.687853,39.746480,50.563249,24.751048,7.389486,48.685068,1.504620,3
3,CF02_B1_C13_M6.wav,CF02,Healthy Control,N/A (Control),241.867490,430.276968,182.607311,77.532756,247.669657,0.030596,...,185.466619,270.086455,2.756198,39.842030,50.198896,25.575913,6.925328,37.504811,1.490399,1
4,CF02_B1_C14_M6.wav,CF02,Healthy Control,N/A (Control),263.730546,370.679797,210.274326,67.002462,160.405471,0.021403,...,149.324023,330.011161,2.476453,37.436137,50.785279,24.886512,5.583488,18.755387,1.670232,1


In [4]:
# STAGE 4 - Acoustic feature statistics: compare Healthy vs Very Low vs Low vs Mid vs High severity groups -...
from src.visualization import plot_praat_feature_comparison, build_praat_group_summary

figure_path = plot_praat_feature_comparison(praat_features, show=True)
group_summary = build_praat_group_summary(praat_features)

summary_path = config.METRICS_DIR / "praat_severity_group_summary.csv"
group_summary.to_csv(summary_path)

print_header("Severity Group Comparison")
print_kv("Figure", figure_path)
print_kv("Group summary table", summary_path)
group_summary


══════════════════════════════════════════════════════════════════════════════
  SEVERITY GROUP COMPARISON
══════════════════════════════════════════════════════════════════════════════
  Figure .................................. C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\figures\praat_severity_comparison.png
  Group summary table ..................... C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\metrics\praat_severity_group_summary.csv


c:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\src\visualization.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


f0_mean                 f0_max                  f0_min  \
                   mean        std        mean         std        mean   
Group_Order                                                              
Healthy      147.116404  47.250786  248.834611  149.140030  106.728158   
Very Low     218.301341  62.426439  380.756641  156.386119  125.232926   
Low          178.288474  51.823904  340.972432  165.915180   94.897452   
Mid          172.808018  45.550263  339.935548  153.838372  108.387628   
High         152.016123  38.404305  214.986783  116.169333  117.741497   

                           f0_std               f0_range              ...  \
                   std       mean        std        mean         std  ...   
Group_Order                                                           ...   
Healthy      31.037873  39.386521  42.519607  142.106454  144.425903  ...   
Very Low     50.105731  60.058354  40.814600  255.523715  158.303726  ...   
Low          29.093046  65.361793  51.153531  246.074980  167.914381  ...   
Mid          27.125030  62.285508  52.348169  231.547920  155.976320  ...   
High         33.454013  26.467448  34.189114   97.245286  115.453537  ...   

            intensity_min           intensity_std           speech_rate  \
                     mean       std          mean       std        mean   
Group_Order                                                               
Healthy         26.311254  1.764622      8.534900  1.430083   34.502270   
Very Low        24.966155  1.926678     10.099223  2.553047   61.677794   
Low             23.796276  1.229928     11.006012  2.314668   41.615166   
Mid             31.186655  7.311106      7.518303  2.521605   77.721520   
High            25.959260  3.181835      9.568885  1.852808   31.522772   

                       pause_duration           voice_breaks             
                   std           mean       std         mean        std  
Group_Order                                                              
Healthy      17.939848       1.340294  0.376138     2.661538   4.039312  
Very Low     32.412087       3.259070  2.855507     9.382353  13.748531  
Low          16.655527       2.545367  1.455246     9.830501  15.226838  
Mid          53.784046       1.818214  1.647155     9.087146  14.784275  
High         26.080779       2.002574  1.412219     3.367320   6.523030  

[5 rows x 62 columns]

In [5]:
# STAGE 5 - Feature correlation analysis.
from src.visualization import plot_feature_correlation

correlation_figure = plot_feature_correlation(praat_features, show=True)
print_header("Feature Correlation")
print_kv("Figure", correlation_figure)


══════════════════════════════════════════════════════════════════════════════
  FEATURE CORRELATION
══════════════════════════════════════════════════════════════════════════════
  Figure .................................. C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\figures\praat_feature_correlation.png


c:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\src\visualization.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# STAGE 6 - Which of those group differences are actually real?
from src.praat import praat_group_significance

significance = praat_group_significance(praat_features)

significance_path = config.METRICS_DIR / "praat_significance.csv"
significance.to_csv(significance_path, index=False)

n_sig = int(significance["significant"].sum())

print_header("Kruskal-Wallis Severity Group Significance")
print_kv("Significance table", significance_path)
print_kv("Significant features (p_adj < 0.05)", f"{n_sig} / {len(significance)}")
print_kv("Strongest separator", significance.iloc[0]["feature"])
significance


══════════════════════════════════════════════════════════════════════════════
  KRUSKAL-WALLIS SEVERITY GROUP SIGNIFICANCE
══════════════════════════════════════════════════════════════════════════════
  Significance table ...................... C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\metrics\praat_significance.csv
  Significant features (p_adj < 0.05) ..... 31 / 31
  Strongest separator ..................... f0_mean


,feature,H,p,p_adj,n_used,significant
0,f0_mean,4333.005600,0.000000e+00,0.000000e+00,21411,True
1,f0_max,3509.794542,0.000000e+00,0.000000e+00,21411,True
2,f0_std,2654.799997,0.000000e+00,0.000000e+00,21411,True
3,f0_range,3506.299834,0.000000e+00,0.000000e+00,21411,True
4,hnr_mean,3007.585155,0.000000e+00,0.000000e+00,21420,True
5,hnr_min,2603.992492,0.000000e+00,0.000000e+00,21420,True
6,pause_duration,7960.870340,0.000000e+00,0.000000e+00,21420,True
7,speech_rate,4186.644083,0.000000e+00,0.000000e+00,21420,True
8,intensity_mean,3606.566005,0.000000e+00,0.000000e+00,21420,True
9,intensity_max,5633.714090,0.000000e+00,0.000000e+00,21420,True


In [7]:
# ===========================================================================
# STAGE 7 - SPEECH-PROCESSING EDA: classical techniques applied directly to this project's own corpus...
from src.training.data import load_manifest

control_row = df_m6[df_m6["Group"] == "Healthy Control"].sample(1, random_state=config.DEFAULT_SEED).iloc[0]
dysarthric_row = df_m6[df_m6["Severity"] == "High"].sample(1, random_state=config.DEFAULT_SEED).iloc[0]

examples = {"Healthy Control": control_row, "Dysarthric (High severity)": dysarthric_row}
for label, row in examples.items():
    print_kv(label, row["Filename"])


  Healthy Control ......................... CM12_B2_CW86_M6.wav
  Dysarthric (High severity) .............. M10_B1_LW_M6.wav


In [8]:
primary_row = dysarthric_row
waveform_np, sr = _load_resampled(primary_row["Filepath"])
waveform_np = waveform_np.squeeze(0).numpy()

fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
t = np.arange(len(waveform_np)) / sr
axes[0].plot(t, waveform_np, color="0.25", linewidth=0.6)
axes[0].set_ylabel("Amplitude")
axes[0].set_title(f"{primary_row['Filename']} — waveform")

freqs, times, mag_db = eda.compute_spectrogram(waveform_np, sr, window_ms=25)
im = axes[1].pcolormesh(times, freqs, mag_db, cmap=SEQUENTIAL_CMAP,
                        vmin=mag_db.max() - 80, vmax=mag_db.max(), shading="auto")
axes[1].set_ylim(0, 5000)
axes[1].set_ylabel("Frequency (Hz)")
axes[1].set_xlabel("Time (s)")
axes[1].set_title("Spectrogram (25 ms window)")
fig.colorbar(im, ax=axes, label="dB", fraction=0.03, pad=0.02)
fig.savefig(config.SIGNAL_FIGURE_DIR / "eda01_production_overview.png", dpi=300, bbox_inches="tight")
plt.show()


C:\Users\surya\AppData\Local\Temp\ipykernel_2632\3511480720.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
from src.praat import extract_suprasegmental_sequence

total_frames = mfcc_frame_count(len(waveform_np))
supra_seq = extract_suprasegmental_sequence(waveform_np, sr, valid_length=len(waveform_np),
                                            total_frames=total_frames)
voicing_mask = supra_seq["voicing"]

_, vad_stats = apply_vad(torch.from_numpy(waveform_np).unsqueeze(0), sr=sr)
speech_start_s = vad_stats["leading_trimmed_s"]
speech_end_s = vad_stats["original_duration_s"] - vad_stats["trailing_trimmed_s"]
print_kv("VAD speech span", f"{speech_start_s:.3f}s - {speech_end_s:.3f}s "
        f"(of {vad_stats['original_duration_s']:.3f}s)")

labels = eda.three_way_segmentation(total_frames, frame_hop_s=0.01,
                                    speech_start_s=speech_start_s, speech_end_s=speech_end_s,
                                    voicing_mask=voicing_mask)
ste_t, ste_v = eda.short_time_energy(waveform_np, sr)
zcr_t, zcr_v = eda.zero_crossing_rate(waveform_np, sr)

eda.plot_voiced_unvoiced_silence(waveform_np, sr, labels, frame_hop_s=0.01,
                                 title=f"{primary_row['Filename']} — VAD + GAD segmentation",
                                 ste=(ste_t, ste_v), zcr=(zcr_t, zcr_v), show=True,
                                 out_name="eda02_voiced_unvoiced_silence.png")

voiced_pct = (labels == 2).mean() * 100
unvoiced_pct = (labels == 1).mean() * 100
silence_pct = (labels == 0).mean() * 100
print_kv("Silence / unvoiced / voiced", f"{silence_pct:.1f}% / {unvoiced_pct:.1f}% / {voiced_pct:.1f}%")


  VAD speech span ......................... 0.770s - 1.438s (of 2.034s)
  Silence / unvoiced / voiced ............. 67.2% / 14.7% / 18.1%


c:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\src\eda.py:390: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
from src.praat import extract_segmental_extra_sequence

vad_waveform, vad_stats2 = apply_vad(torch.from_numpy(waveform_np).unsqueeze(0), sr=sr)
vad_waveform_np = vad_waveform.squeeze(0).numpy()
seg_total_frames = mfcc_frame_count(len(vad_waveform_np))
extra_seq = extract_segmental_extra_sequence(vad_waveform_np, sr, valid_length=len(vad_waveform_np),
                                             total_frames=seg_total_frames)

freqs_n, times_n, mag_db_n = eda.compute_spectrogram(vad_waveform_np, sr, window_ms=30)
fig, ax = plt.subplots(figsize=(9, 4.5))
im = ax.pcolormesh(times_n, freqs_n, mag_db_n, cmap=SEQUENTIAL_CMAP,
                   vmin=mag_db_n.max() - 80, vmax=mag_db_n.max(), shading="auto")
ax.set_ylim(0, 4000)

frame_times = np.arange(seg_total_frames) * 0.01
for name, arr in (("F1", extra_seq["f1_hz"]), ("F2", extra_seq["f2_hz"]), ("F3", extra_seq["f3_hz"])):
    valid = arr > 0
    ax.scatter(frame_times[valid], arr[valid], s=6, color=FORMANT_COLORS[name], label=name)

ax.legend(loc="upper right", fontsize=9)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Frequency (Hz)")
ax.set_title(f"{primary_row['Filename']} — formant tracks on narrowband spectrogram")
fig.colorbar(im, ax=ax, label="dB", fraction=0.03, pad=0.02)
fig.savefig(config.SIGNAL_FIGURE_DIR / "eda03_formant_tracks.png", dpi=300, bbox_inches="tight")
plt.show()


C:\Users\surya\AppData\Local\Temp\ipykernel_2632\683562870.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6), sharex="col")
for col, (label, row) in enumerate(examples.items()):
    wav, sr_ex = _load_resampled(row["Filepath"])
    wav = wav.squeeze(0).numpy()
    t_ste, v_ste = eda.short_time_energy(wav, sr_ex)
    t_zcr, v_zcr = eda.zero_crossing_rate(wav, sr_ex)
    axes[0, col].plot(t_ste, v_ste, color="#4c72b0")
    axes[0, col].set_title(label)
    axes[1, col].plot(t_zcr, v_zcr, color="#dd8452")
axes[0, 0].set_ylabel("Short-time energy")
axes[1, 0].set_ylabel("Zero-crossing rate")
for ax in axes[1, :]:
    ax.set_xlabel("Time (s)")
fig.suptitle("Short-time time-domain parameters — Control vs. Dysarthric")
fig.tight_layout()
fig.savefig(config.SIGNAL_FIGURE_DIR / "eda04_time_domain_parameters.png", dpi=300, bbox_inches="tight")
plt.show()


C:\Users\surya\AppData\Local\Temp\ipykernel_2632\2383639682.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
t_sc, centroid, rolloff = eda.spectral_centroid_rolloff(waveform_np, sr)
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(t_sc, centroid, color="#55a868", label="Spectral centroid")
ax.plot(t_sc, rolloff, color="#c44e52", linestyle="--", label="Spectral rolloff (85%)")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Frequency (Hz)")
ax.set_title(f"{primary_row['Filename']} — short-time frequency-domain parameters")
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(config.SIGNAL_FIGURE_DIR / "eda05_frequency_domain_parameters.png", dpi=300, bbox_inches="tight")
plt.show()


C:\Users\surya\AppData\Local\Temp\ipykernel_2632\1476189435.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
eda.plot_wideband_narrowband_spectrogram(vad_waveform_np, sr,
                                          title=f"{primary_row['Filename']} — wideband vs. narrowband",
                                          show=True, out_name="eda06_wideband_narrowband.png")


'C:\\Users\\surya\\OneDrive\\Desktop\\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\\outputs\\figures\\signals\\eda06_wideband_narrowband.png'

In [14]:
voiced_frame_times = frame_times[extra_seq["f1_hz"] > 0]
frame_center = float(voiced_frame_times[len(voiced_frame_times) // 2]) if len(voiced_frame_times) else 0.2

eda.plot_cepstral_analysis(vad_waveform_np, sr, frame_center_s=frame_center,
                           title=f"{primary_row['Filename']} — cepstral analysis at t={frame_center:.2f}s",
                           show=True, out_name="eda07_cepstral_analysis.png")


'C:\\Users\\surya\\OneDrive\\Desktop\\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\\outputs\\figures\\signals\\eda07_cepstral_analysis.png'

In [15]:
from src.preprocessing import extract_mfcc_features_cached

mfcc_tensor = extract_mfcc_features_cached(primary_row["Filepath"]).squeeze(0)  # (39, T): MFCC+delta+delta-delta
mfcc_np = mfcc_tensor[:config.N_MFCC].numpy()  # static coefficients only, for the classic MFCC heatmap

fig, ax = plt.subplots(figsize=(9, 3.5))
im = ax.imshow(mfcc_np, aspect="auto", origin="lower", cmap=SEQUENTIAL_CMAP,
               extent=[0, mfcc_np.shape[1] * 0.01, 0, config.N_MFCC])
ax.set_xlabel("Time (s)")
ax.set_ylabel("MFCC coefficient")
ax.set_title(f"{primary_row['Filename']} — MFCC ({config.N_MFCC} coefficients)")
fig.colorbar(im, ax=ax, label="Coefficient value", fraction=0.03, pad=0.02)
fig.tight_layout()
fig.savefig(config.SIGNAL_FIGURE_DIR / "eda08_mfcc.png", dpi=300, bbox_inches="tight")
plt.show()


C:\Users\surya\AppData\Local\Temp\ipykernel_2632\4214668279.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
eda.plot_lpc_envelope(vad_waveform_np, sr, frame_center_s=frame_center, order=16,
                      title=f"{primary_row['Filename']} — LPC envelope (order 16) at t={frame_center:.2f}s",
                      show=True, out_name="eda09_lpc_envelope.png")


'C:\\Users\\surya\\OneDrive\\Desktop\\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\\outputs\\figures\\signals\\eda09_lpc_envelope.png'

In [17]:
# ===========================================================================
# STAGE 17 - FEATURE-EXTRACTION SUMMARY: formal inventory of what each branch of...
model = build_model(SEVERITY_MODEL_NAME, config.NUM_CLASSES["severity"], num_speakers=2)
audit = feature_audit(model=model, num_classes=config.NUM_CLASSES["severity"])

print_header("Feature-extraction summary")
print_kv("Model", SEVERITY_MODEL_NAME)
print_kv("Branches", 3)
print_kv("Fused representation", f"{audit['fusion']['fused_dim']}-dim")


Loading weights: 100%|██████████| 210/210 [00:00<00:00, 4536.88it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key            | Status     |  | 
---------------+------------+--+-
lm_head.bias   | UNEXPECTED |  | 
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Wav2Vec2Model does not expose input embeddings. Gradients cannot flow back to the token embeddings when using adapters or gradient checkpointing. Override `get_input_embeddings` to fully support those features, or set `_input_embed_layer` to the attribute name that holds the embeddings.



══════════════════════════════════════════════════════════════════════════════
  FEATURE-EXTRACTION SUMMARY
══════════════════════════════════════════════════════════════════════════════
  Model ................................... gated_fusion_three_branch
  Branches ................................ 3
  Fused representation .................... 256-dim


In [18]:
branch_inventory = pd.DataFrame([
    {
        "branch": "Learned",
        "input_representation": "Raw waveform (16 kHz, 4.0 s window)",
        "extraction_method": f"{audit['learned_branch']['wav2vec2_model']} "
                             f"+ LoRA (rank {audit['learned_branch']['lora_rank']}, "
                             f"alpha {audit['learned_branch']['lora_alpha']}) on "
                             f"{', '.join(audit['learned_branch']['lora_target_modules'])}",
        "raw_dim": config.WAV2VEC_EMBED_DIM,
        "bottleneck_dim": audit["learned_branch"]["dimensions"],
        "source_module": "src/models/deep_pathway.py (DeepPathway)",
        "trainable": "Partial — LoRA adapters + projection only",
    },
    {
        "branch": "Segmental",
        "input_representation": "MFCC + \u0394 + \u0394\u0394 + framewise formants (F1-F3) + framewise HNR",
        "extraction_method": "3-layer 1D-CNN + masked mean-pool",
        "raw_dim": audit["segmental_branch"]["input_channels"],
        "bottleneck_dim": audit["segmental_branch"]["dimensions"],
        "source_module": "src/models/segmental_pathway.py (SegmentalPathway)",
        "trainable": "Fully trainable (CNN + bottleneck)",
    },
    {
        "branch": "Suprasegmental",
        "input_representation": "F0 (semitones) + voicing mask + intensity (dB)",
        "extraction_method": "2-layer 1D-CNN + masked mean-pool",
        "raw_dim": audit["suprasegmental_branch"]["input_channels"],
        "bottleneck_dim": audit["suprasegmental_branch"]["dimensions"],
        "source_module": "src/models/suprasegmental_pathway.py (SuprasegmentalPathway)",
        "trainable": "Fully trainable (CNN + bottleneck)",
    },
])
branch_inventory


,branch,input_representation,extraction_method,raw_dim,bottleneck_dim,source_module,trainable
0,Learned,"Raw waveform (16 kHz, 4.0 s window)","facebook/wav2vec2-base-960h + LoRA (rank 8, al...",768,128,src/models/deep_pathway.py (DeepPathway),Partial — LoRA adapters + projection only
1,Segmental,MFCC + Δ + ΔΔ + framewise formants (F1-F3) + f...,3-layer 1D-CNN + masked mean-pool,43,64,src/models/segmental_pathway.py (SegmentalPath...,Fully trainable (CNN + bottleneck)
2,Suprasegmental,F0 (semitones) + voicing mask + intensity (dB),2-layer 1D-CNN + masked mean-pool,3,64,src/models/suprasegmental_pathway.py (Supraseg...,Fully trainable (CNN + bottleneck)


In [19]:
branch_inventory.to_csv(config.TABLES_DIR / "feature_branch_inventory.csv", index=False)
print_kv("Saved", config.TABLES_DIR / "feature_branch_inventory.csv")


  Saved ................................... C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\tables\feature_branch_inventory.csv


In [20]:
param_table = architecture_table(model, model_name=SEVERITY_MODEL_NAME)
param_table_display = param_table.copy()
for c in ("trainable_params", "total_params", "frozen_params"):
    param_table_display[c] = param_table_display[c].map(lambda v: f"{v:,}")
param_table_display["pct_trainable"] = param_table_display["pct_trainable"].map(lambda v: f"{v:.2f}%")
param_table_display


,submodule,trainable_params,total_params,frozen_params,pct_trainable
0,deep_pathway,"442,368","94,813,312","94,370,944",0.47%
1,learned_projection,"98,688","98,688",0,100.00%
2,segmental_pathway,"113,088","113,088",0,100.00%
3,suprasegmental_pathway,"15,168","15,168",0,100.00%
4,gate,"8,323","8,323",0,100.00%
5,classifier,259,259,0,100.00%
6,speaker_head,"66,306","66,306",0,100.00%
7,TOTAL,"744,200","95,115,144","94,370,944",0.78%


In [21]:
param_table.to_csv(config.TABLES_DIR / "feature_branch_parameter_counts.csv", index=False)
print_kv("Saved", config.TABLES_DIR / "feature_branch_parameter_counts.csv")


  Saved ................................... C:\Users\surya\OneDrive\Desktop\Acoustic-Aware-Fusion-Architecture-for-Dysarthria-Classification\outputs\tables\feature_branch_parameter_counts.csv


In [22]:
import matplotlib.pyplot as plt

fused = audit["fusion"]
branches = ["learned", "segmental", "supra"]
dims = [fused["learned_dim"], fused["segmental_dim"], fused["supra_dim"]]
labels = ["Learned\n(wav2vec2+LoRA)", "Segmental\n(MFCC+formants+HNR)", "Suprasegmental\n(F0+voicing+intensity)"]
colors = [BRANCH_COLORS[b] for b in branches]

fig, ax = plt.subplots(figsize=(7, 2.2))
left = 0
for dim, label, color in zip(dims, labels, colors):
    ax.barh(0, dim, left=left, color=color, edgecolor="white", height=0.6)
    ax.text(left + dim / 2, 0, f"{label}\n{dim}D", ha="center", va="center", fontsize=8.5, color="white")
    left += dim
ax.set_xlim(0, fused["fused_dim"])
ax.set_yticks([])
ax.set_xlabel(f"Z_unified dimension (total {fused['fused_dim']}D)")
ax.set_title("Fused embedding composition")
fig.tight_layout()
fig.savefig(config.FIGURE_DIR / "feature_zunified_composition.png", dpi=300, bbox_inches="tight")
plt.show()


C:\Users\surya\AppData\Local\Temp\ipykernel_2632\775276300.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [23]:
families = audit["segmental_branch"]["engineered_feature_families"]
family_names, family_counts = [], []
for entry in families:
    name, count_str = entry.rsplit("(", 1)
    family_names.append(name.strip())
    family_counts.append(int(count_str.rstrip(")")))

assert sum(family_counts) == audit["segmental_branch"]["input_channels"]

fig, ax = plt.subplots(figsize=(7, 4))
bar_colors = plt.get_cmap("viridis")([i / max(1, len(family_names) - 1) for i in range(len(family_names))])
bars = ax.bar(family_names, family_counts, color=bar_colors, edgecolor="white")
ax.bar_label(bars, padding=2)
ax.set_ylabel("Channels")
ax.set_title(f"Segmental branch input channels (total {sum(family_counts)})")
plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
fig.tight_layout()
fig.savefig(config.FIGURE_DIR / "feature_segmental_channel_breakdown.png", dpi=300, bbox_inches="tight")
plt.show()


C:\Users\surya\AppData\Local\Temp\ipykernel_2632\794703954.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [24]:
supra_families = audit["suprasegmental_branch"]["engineered_feature_families"]
supra_table = pd.DataFrame({"channel": range(1, len(supra_families) + 1), "feature": supra_families})
supra_table


,channel,feature
0,1,F0 (semitones; zero when unvoiced)
1,2,voicing mask
2,3,intensity (dB)
